# alternating soundsource analysis

In [1]:
from scipy.io import loadmat
from scipy.signal import butter, filtfilt
import numpy as np
import pandas as pd
import polars as pl
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
from workbench.data.preprocess import TriggRasterPY, combTableCreate, processTableRow
import h5py
import tables as tb

In [2]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT * FROM Recordings WHERE (Animal_Id, Cell_Id) IN (
SELECT Animal_Id, Cell_Id FROM Recordings WHERE Condition IN ("Baseline", "soso") GROUP BY Animal_Id, Cell_Id 
HAVING COUNT(DISTINCT Condition) >= 2) 
AND Condition IN ("Baseline","soso") AND use = 1 AND Folders_generated = 1
"""

datatable= pd.read_sql_query(sql, conn)
conn.close()

In [3]:
# compute the preprocessed data from the rawdata
#datapath = r"\\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables"
#filename = "soso_comb.5"
#try:
#    print('hi')
#except:
#    Warning("comb_table does not exist. Creating the file....")
#    comb_table = combTableCreate(datatable, datapath, filename)


In [4]:
d_path = r"\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data8\Baseline\exp_data.mat"
exp = loadmat(
    d_path,
    struct_as_record=False,
    squeeze_me=False,
    simplify_cells= True  
)

In [5]:
processed_data = exp.get('processed_data', None)
raw_data = exp.get('raw_data', None)
spkT = processed_data['spike_sorting_data']['spike_times']
videoT = raw_data['ephys_data']['ttl_times']
angles = processed_data['tracking_data']['angles']

In [6]:
comb_table = combTableCreate(datatable, "", "")

----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 4, Baseline being processed
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 4, soso being processed
----------------------------------------------------------------------------------------------------
\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data4\soso\exp_data.mat
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 6, Baseline being processed
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 6, soso b

In [ ]:
comb_df = pd.DataFrame(comb_table)
comb_df = pl.from_pandas(comb_df)
comb_df = write_parquet(comb_df, r"Z:\lab share\Data\Florian\comb_tables\soso_comb.parquet")
#comb_df = comb_df.to_parquet(r"Z:\lab share\Data\Florian\comb_tables\soso_comb.parquet", index=False)

# import tables as tb
# h5 = tb.open_file("filename.h5")
# tbl = h5.root.data.table
# arr = table.read() # -> numpy structured array
# df = pd.DataFrame.from_records(arr)
# pldf = pl.from_pandas(df)
# h5.close()

ArrowInvalid: Can only convert 1-dimensional array values